# Construction du dataset maître CollectionLens

## Objectif

Construire le premier dataset maître CollectionLens à partir des données agrégées.

Chaque ISBN du benchmark étendu doit être transformé en une fiche unique résultant de l'application de la stratégie multi-sources.

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

sys.path.append(
    str(PROJECT_ROOT / "src")
)

In [3]:
from collectionlens.pipelines.metadata_aggregation import (
    aggregate_book_metadata,
)

In [4]:
RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pipeline_quality_baseline"
)

In [5]:
isbn_list = [
    path.stem
    for path in (
        RAW_DIR
        / "nudger"
    ).glob("*.json")
]

len(isbn_list)

1036

In [6]:
master_rows = []

In [7]:
for isbn in isbn_list:

    sources_data = {}

    for source_name in [
        "nudger",
        "google_books",
        "bnf",
        "openlibrary",
    ]:

        json_path = (
            RAW_DIR
            / source_name
            / f"{isbn}.json"
        )

        with json_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            payload = json.load(file)

        sources_data[source_name] = payload["result"]

    aggregated_book = aggregate_book_metadata(
        sources_data=sources_data,
    )

    master_rows.append(
        aggregated_book
    )

In [8]:
df_master = pd.DataFrame(
    master_rows
)

df_master.shape

(1036, 21)

In [9]:
df_master.head()

,isbn,title,title_source,authors,authors_source,publisher,publisher_source,published_date,published_date_source,description,...,categories,categories_source,page_count,page_count_source,format,format_source,cover_url,cover_url_source,bnf_ark,bnf_ark_source
0,9782351420225,Satan 666,google_books,[Seishi Kishimoto],google_books,Kurokawa (Paris),bnf,2005-10-13,google_books,Collection : Collection dirigée par Grégoire H...,...,None,None,184,google_books,"1 vol. (184 p.) : ill., couv. ill., jaquette i...",bnf,None,None,http://catalogue.bnf.fr/ark:/12148/cb40085004q,bnf
1,9782845800717,L'aube,google_books,[Osamu Tezuka],google_books,Éd. Tonkam (Paris),bnf,2007-02-07,google_books,Le phénix est un oiseau immortel ! Une créatur...,...,None,None,339,google_books,"339 p. : ill., jaquette ill. en coul. ; 17 cm",bnf,None,None,http://catalogue.bnf.fr/ark:/12148/cb376291158,bnf
2,2910635169,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,9781421532271,"Color Bleach, Bleach",nudger,[Tite Kubo],google_books,[Viz Media],nudger,2010-08-10,google_books,The Soul Reaper's Handbook This indispensable ...,...,[Comics & Graphic Novels],google_books,184,nudger,Broché,nudger,http://books.google.com/books/content?id=I6Idn...,google_books,None,None
4,9782016284407,Laïcité,nudger,[Charb (1967-2015). Auteur du texte],bnf,[Robinson],nudger,2025-01-02,google_books,"Je comprends qu&#39;on puisse être juif, chrét...",...,None,None,96,nudger,Album,nudger,None,None,http://catalogue.bnf.fr/ark:/12148/cb476267290,bnf


In [10]:
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1036 entries, 0 to 1035
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   isbn                   1036 non-null   object
 1   title                  1000 non-null   object
 2   title_source           1000 non-null   object
 3   authors                850 non-null    object
 4   authors_source         850 non-null    object
 5   publisher              982 non-null    object
 6   publisher_source       982 non-null    object
 7   published_date         894 non-null    object
 8   published_date_source  894 non-null    object
 9   description            822 non-null    object
 10  description_source     822 non-null    object
 11  categories             76 non-null     object
 12  categories_source      76 non-null     object
 13  page_count             984 non-null    object
 14  page_count_source      984 non-null    object
 15  format               

In [11]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "master_dataset"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [12]:
csv_path = (
    OUTPUT_DIR
    / "collectionlens_master.csv"
)

parquet_path = (
    OUTPUT_DIR
    / "collectionlens_master.parquet"
)

In [13]:
df_master.to_csv(
    csv_path,
    index=False,
)

In [14]:
print(csv_path)
print(parquet_path)

c:\Users\yoann\Documents\mes projets\CollectionLens V2\data\processed\master_dataset\collectionlens_master.csv
c:\Users\yoann\Documents\mes projets\CollectionLens V2\data\processed\master_dataset\collectionlens_master.parquet


# Conclusion

Le dataset maître CollectionLens a été généré avec succès.

Chaque ouvrage du benchmark étendu dispose désormais d'une fiche unique agrégée à partir des différentes sources bibliographiques.

Ce dataset constitue désormais la base de référence pour les futures analyses qualité, la gestion de collection utilisateur et le moteur de recommandation.